In [3]:

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind

warnings.filterwarnings("ignore")


desktop = r"C:\Users\DELL\Desktop"

data_file = os.path.join(desktop, "TZHR82FL.DTA")


try:
    df = pd.read_stata(data_file, convert_categoricals=False)
    print("Dataset loaded successfully.")
except Exception as e:
    raise Exception(f"Unable to load dataset:\n{e}")



df.columns = df.columns.str.lower().str.strip()


required_household = [
    "hv001",
    "hv002",
    "hv219"
]


required_prefixes = [
    "hvidx",
    "hv101",
    "hv102",
    "hv104",
    "hv105"
]

# ------------------------------------------------------------
# Check Household Variables
# ------------------------------------------------------------

missing_household = [
    x for x in required_household
    if x not in df.columns
]

if len(missing_household) > 0:
    raise ValueError(
        f"Missing household variables: {missing_household}"
    )

# ------------------------------------------------------------
# Check Member Variables
# ------------------------------------------------------------

for prefix in required_prefixes:

    vars_found = [
        c for c in df.columns
        if c.startswith(prefix + "_")
    ]

    if len(vars_found) == 0:
        raise ValueError(
            f"No variables found beginning with '{prefix}_'"
        )

print("All required variables detected.")

# ------------------------------------------------------------
# Replace DHS Missing Codes
# ------------------------------------------------------------

missing_codes = [
    97,
    98,
    99,
    997,
    998,
    999,
    9997,
    9998,
    9999
]

df.replace(missing_codes, np.nan, inplace=True)

# ------------------------------------------------------------
# Convert Numeric Columns Safely
# ------------------------------------------------------------

for col in df.columns:

    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

# ------------------------------------------------------------
# Household Head Sex
# ------------------------------------------------------------

sex_map = {
    1: "Male",
    2: "Female"
}

df["head_sex"] = df["hv219"].map(sex_map)

# ------------------------------------------------------------
# Household Identifier
# ------------------------------------------------------------

df["household_id"] = (
    df["hv001"].astype(str)
    + "_"
    + df["hv002"].astype(str)
)

# ------------------------------------------------------------
# Dataset Summary
# ------------------------------------------------------------

print("\nDataset Summary")
print("---------------------------")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Unique households: {df['household_id'].nunique():,}")

print("\nHousehold Head Sex Distribution")
print(df["head_sex"].value_counts(dropna=False))

print("\nPART 1 COMPLETED SUCCESSFULLY")

Dataset loaded successfully.
All required variables detected.

Dataset Summary
---------------------------
Rows: 15,705
Columns: 9,248
Unique households: 15,685

Household Head Sex Distribution
head_sex
Male      11182
Female     4523
Name: count, dtype: int64

PART 1 COMPLETED SUCCESSFULLY


In [7]:
# ============================================================
# PART 2: RESHAPE HOUSEHOLD ROSTER (WIDE → LONG)
# ============================================================

print("\nReshaping household roster...")

member_records = []

member_numbers = []

for c in df.columns:
    if c.startswith("hv105_"):
        try:
            member_numbers.append(int(c.split("_")[1]))
        except:
            pass

member_numbers = sorted(member_numbers)

for member in member_numbers:

    suffix = f"{member:02d}"

    age_col = f"hv105_{suffix}"
    sex_col = f"hv104_{suffix}"
    rel_col = f"hv101_{suffix}"
    res_col = f"hv102_{suffix}"
    idx_col = f"hvidx_{suffix}"

    if age_col not in df.columns:
        continue

    temp = pd.DataFrame({
        "household_id": df["household_id"],
        "hv001": df["hv001"],
        "hv002": df["hv002"],
        "head_sex": df["head_sex"],
        "member_no": member,
        "line_number": df[idx_col] if idx_col in df.columns else np.nan,
        "relationship": df[rel_col] if rel_col in df.columns else np.nan,
        "usual_resident": df[res_col] if res_col in df.columns else np.nan,
        "sex": df[sex_col] if sex_col in df.columns else np.nan,
        "age": df[age_col]
    })

    member_records.append(temp)

roster_long = pd.concat(member_records, ignore_index=True)

# ------------------------------------------------------------
# Remove Empty Records
# ------------------------------------------------------------

roster_long = roster_long[
    ~(roster_long["age"].isna() &
      roster_long["sex"].isna() &
      roster_long["relationship"].isna())
].copy()

# ------------------------------------------------------------
# Convert Numeric Variables
# ------------------------------------------------------------

numeric_cols = [
    "line_number",
    "relationship",
    "usual_resident",
    "sex",
    "age"
]

for col in numeric_cols:
    roster_long[col] = pd.to_numeric(
        roster_long[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# Convert Codes to Labels
# ------------------------------------------------------------

roster_long["sex"] = roster_long["sex"].replace({
    1: "Male",
    2: "Female"
})

relationship_labels = {
    1: "Head",
    2: "Spouse",
    3: "Child",
    4: "Child-in-law",
    5: "Grandchild",
    6: "Parent",
    7: "Parent-in-law",
    8: "Brother/Sister",
    9: "Other Relative",
    10: "Adopted/Foster Child",
    11: "Not Related",
    98: "Don't Know"
}

roster_long["relationship"] = (
    roster_long["relationship"]
    .replace(relationship_labels)
)

resident_labels = {
    1: "Usual Resident",
    0: "Visitor"
}

roster_long["usual_resident"] = (
    roster_long["usual_resident"]
    .replace(resident_labels)
)

# ------------------------------------------------------------
# Sort Records
# ------------------------------------------------------------

roster_long.sort_values(
    ["household_id", "member_no"],
    inplace=True
)

roster_long.reset_index(
    drop=True,
    inplace=True
)

# ------------------------------------------------------------
# Save Household Roster
# ------------------------------------------------------------

roster_file = os.path.join(
    desktop,
    "Household_Roster_Long.xlsx"
)

roster_long.to_excel(
    roster_file,
    index=False
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nHousehold roster successfully reshaped.")

print(f"Total individuals: {len(roster_long):,}")

print(f"Total households: {roster_long['household_id'].nunique():,}")

print("\nFirst five records:")

print(roster_long.head())

print(f"\nRoster saved to:\n{roster_file}")

print("\nPART 2 COMPLETED SUCCESSFULLY")


Reshaping household roster...

Household roster successfully reshaped.
Total individuals: 73,774
Total households: 15,685

First five records:
  household_id  hv001  hv002 head_sex  member_no  line_number relationship  \
0      1.0_1.0    1.0    1.0     Male          1          1.0         Head   
1      1.0_1.0    1.0    1.0     Male          2          2.0       Spouse   
2      1.0_1.0    1.0    1.0     Male          3          3.0   Grandchild   
3     1.0_12.0    1.0   12.0     Male          1          1.0         Head   
4     1.0_12.0    1.0   12.0     Male          2          2.0       Spouse   

   usual_resident     sex   age  
0  Usual Resident    Male  62.0  
1  Usual Resident  Female  59.0  
2  Usual Resident  Female  10.0  
3  Usual Resident    Male  76.0  
4  Usual Resident  Female  59.0  

Roster saved to:
C:\Users\DELL\Desktop\Household_Roster_Long.xlsx

PART 2 COMPLETED SUCCESSFULLY


In [8]:
print(type(roster_long))

<class 'pandas.core.frame.DataFrame'>


In [9]:
# ============================================================
# PART 3: CLEAN DATA & COMPUTE HOUSEHOLD DEPENDENCY RATIO
# ============================================================

print("\nCleaning household roster...")

# ------------------------------------------------------------
# Keep only usual residents
# ------------------------------------------------------------

if "usual_resident" in roster_long.columns:
    roster_long = roster_long[
        roster_long["usual_resident"] == "Usual Resident"
    ].copy()

# ------------------------------------------------------------
# Clean age variable
# ------------------------------------------------------------

roster_long["age"] = pd.to_numeric(
    roster_long["age"],
    errors="coerce"
)

roster_long = roster_long[
    roster_long["age"].notna()
]

roster_long = roster_long[
    (roster_long["age"] >= 0) &
    (roster_long["age"] <= 120)
]

# ------------------------------------------------------------
# Create age-group indicators
# ------------------------------------------------------------

roster_long["child"] = (
    roster_long["age"] <= 14
).astype(int)

roster_long["working_age"] = (
    (roster_long["age"] >= 15) &
    (roster_long["age"] <= 64)
).astype(int)

roster_long["elderly"] = (
    roster_long["age"] >= 65
).astype(int)

roster_long["dependents"] = (
    roster_long["child"] +
    roster_long["elderly"]
)

# ------------------------------------------------------------
# Household-level aggregation
# ------------------------------------------------------------

household = (
    roster_long
    .groupby("household_id")
    .agg(
        hv001=("hv001", "first"),
        hv002=("hv002", "first"),
        head_sex=("head_sex", "first"),
        household_size=("member_no", "count"),
        children=("child", "sum"),
        working_age=("working_age", "sum"),
        elderly=("elderly", "sum"),
        dependents=("dependents", "sum")
    )
    .reset_index()
)

# ------------------------------------------------------------
# Dependency Ratio
# ------------------------------------------------------------

household["dependency_ratio"] = np.where(
    household["working_age"] > 0,
    (household["dependents"] /
     household["working_age"]) * 100,
    np.nan
)

household["dependency_ratio"] = household[
    "dependency_ratio"
].round(2)

# ------------------------------------------------------------
# Remove impossible households
# ------------------------------------------------------------

household = household[
    household["household_size"] > 0
].copy()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nDependency Ratio Summary")

print(
    household["dependency_ratio"]
    .describe()
)

# ------------------------------------------------------------
# Save cleaned household data
# ------------------------------------------------------------

cleaned_file = os.path.join(
    desktop,
    "Cleaned_Household_Data.xlsx"
)

household.to_excel(
    cleaned_file,
    index=False
)

print(f"\nSaved:\n{cleaned_file}")

print(f"\nHouseholds analysed: {len(household):,}")

print(f"Individuals analysed: {len(roster_long):,}")

print("\nPART 3 COMPLETED SUCCESSFULLY")


Cleaning household roster...

Dependency Ratio Summary
count    14931.000000
mean       106.194622
std         99.165452
min          0.000000
25%         40.000000
50%        100.000000
75%        150.000000
max        900.000000
Name: dependency_ratio, dtype: float64

Saved:
C:\Users\DELL\Desktop\Cleaned_Household_Data.xlsx

Households analysed: 15,681
Individuals analysed: 72,133

PART 3 COMPLETED SUCCESSFULLY


In [10]:
# ============================================================
# PART 4: DESCRIPTIVE STATISTICS & INDEPENDENT T-TEST
# ============================================================

print("\nRunning statistical analysis...")

# ------------------------------------------------------------
# Overall Statistics
# ------------------------------------------------------------

overall_stats = household[[
    "dependency_ratio"
]].describe().T

overall_stats.rename(columns={
    "count": "Number_of_Households",
    "mean": "Mean",
    "std": "Standard_Deviation",
    "min": "Minimum",
    "25%": "Q1",
    "50%": "Median",
    "75%": "Q3",
    "max": "Maximum"
}, inplace=True)

# ------------------------------------------------------------
# Grouped Statistics
# ------------------------------------------------------------

grouped_stats = (
    household
    .groupby("head_sex")["dependency_ratio"]
    .agg(
        Number_of_Households="count",
        Mean="mean",
        Standard_Deviation="std",
        Minimum="min",
        Median="median",
        Maximum="max"
    )
    .reset_index()
)

grouped_stats["Mean"] = grouped_stats["Mean"].round(2)
grouped_stats["Standard_Deviation"] = grouped_stats["Standard_Deviation"].round(2)
grouped_stats["Minimum"] = grouped_stats["Minimum"].round(2)
grouped_stats["Median"] = grouped_stats["Median"].round(2)
grouped_stats["Maximum"] = grouped_stats["Maximum"].round(2)

# ------------------------------------------------------------
# Male vs Female Households
# ------------------------------------------------------------

male = household.loc[
    household["head_sex"] == "Male",
    "dependency_ratio"
].dropna()

female = household.loc[
    household["head_sex"] == "Female",
    "dependency_ratio"
].dropna()

# ------------------------------------------------------------
# Independent Samples T-test
# ------------------------------------------------------------

t_stat, p_value = ttest_ind(
    male,
    female,
    equal_var=False,
    nan_policy="omit"
)

# ------------------------------------------------------------
# Report Table
# ------------------------------------------------------------

report_table = grouped_stats.copy()

report_table["Variable"] = "Dependency Ratio (%)"

report_table = report_table[
    [
        "Variable",
        "head_sex",
        "Number_of_Households",
        "Mean",
        "Standard_Deviation",
        "Minimum",
        "Median",
        "Maximum"
    ]
]

# ------------------------------------------------------------
# Save Excel Outputs
# ------------------------------------------------------------

overall_file = os.path.join(
    desktop,
    "Overall_Statistics.xlsx"
)

group_file = os.path.join(
    desktop,
    "Grouped_Statistics.xlsx"
)

report_file = os.path.join(
    desktop,
    "Report_Table.xlsx"
)

overall_stats.to_excel(
    overall_file
)

grouped_stats.to_excel(
    group_file,
    index=False
)

report_table.to_excel(
    report_file,
    index=False
)

# ------------------------------------------------------------
# Save T-test Results
# ------------------------------------------------------------

ttest_file = os.path.join(
    desktop,
    "T_Test_Results.txt"
)

with open(ttest_file, "w") as f:

    f.write("INDEPENDENT SAMPLES T-TEST\n")
    f.write("=" * 40 + "\n\n")

    f.write(f"Male Households: {len(male)}\n")
    f.write(f"Female Households: {len(female)}\n\n")

    f.write(f"Male Mean Dependency Ratio: {male.mean():.2f}\n")
    f.write(f"Female Mean Dependency Ratio: {female.mean():.2f}\n\n")

    f.write(f"T-statistic: {t_stat:.4f}\n")
    f.write(f"P-value: {p_value:.6f}\n")

    if p_value < 0.05:
        f.write("\nResult: Significant difference (p < 0.05)")
    else:
        f.write("\nResult: No significant difference (p >= 0.05)")

# ------------------------------------------------------------
# Display Results
# ------------------------------------------------------------

print("\nGrouped Statistics")
print(grouped_stats)

print("\nIndependent Samples T-test")

print(f"T-statistic : {t_stat:.4f}")
print(f"P-value     : {p_value:.6f}")

print("\nFiles Saved")

print(overall_file)
print(group_file)
print(report_file)
print(ttest_file)

print("\nPART 4 COMPLETED SUCCESSFULLY")


Running statistical analysis...

Grouped Statistics
  head_sex  Number_of_Households    Mean  Standard_Deviation  Minimum  Median  \
0   Female                  4094  124.07              121.43      0.0   100.0   
1     Male                 10837   99.44               88.39      0.0   100.0   

   Maximum  
0    900.0  
1    900.0  

Independent Samples T-test
T-statistic : -11.8472
P-value     : 0.000000

Files Saved
C:\Users\DELL\Desktop\Overall_Statistics.xlsx
C:\Users\DELL\Desktop\Grouped_Statistics.xlsx
C:\Users\DELL\Desktop\Report_Table.xlsx
C:\Users\DELL\Desktop\T_Test_Results.txt

PART 4 COMPLETED SUCCESSFULLY


In [9]:
# ============================================================
# PART 5: VISUALIZATION & FINAL SUMMARY
# ============================================================

print("\nGenerating visualizations...")

# ------------------------------------------------------------
# Plot 1: Average Dependency Ratio by Household Head Sex
# ------------------------------------------------------------

plt.figure(figsize=(7,5))

mean_values = (
    household.groupby("head_sex")["dependency_ratio"]
    .mean()
)

plt.bar(
    mean_values.index,
    mean_values.values
)

plt.title("Average Household Dependency Ratio by Sex of Household Head")
plt.xlabel("Sex of Household Head")
plt.ylabel("Average Dependency Ratio (%)")
plt.grid(axis="y", alpha=0.3)

bar_file = os.path.join(
    desktop,
    "Average_Dependency_Ratio.png"
)

plt.tight_layout()
plt.savefig(bar_file, dpi=300)
plt.close()

# ------------------------------------------------------------
# Plot 2: Histogram
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.hist(
    household["dependency_ratio"].dropna(),
    bins=20
)

plt.title("Distribution of Household Dependency Ratios")
plt.xlabel("Dependency Ratio (%)")
plt.ylabel("Frequency")
plt.grid(alpha=0.3)

hist_file = os.path.join(
    desktop,
    "Dependency_Histogram.png"
)

plt.tight_layout()
plt.savefig(hist_file, dpi=300)
plt.close()

# ------------------------------------------------------------
# Plot 3: Boxplot
# ------------------------------------------------------------

male_data = household.loc[
    household["head_sex"]=="Male",
    "dependency_ratio"
].dropna()

female_data = household.loc[
    household["head_sex"]=="Female",
    "dependency_ratio"
].dropna()

plt.figure(figsize=(7,5))

plt.boxplot(
    [male_data, female_data],
    labels=["Male","Female"]
)

plt.title("Dependency Ratio by Household Head Sex")
plt.xlabel("Household Head Sex")
plt.ylabel("Dependency Ratio (%)")
plt.grid(axis="y", alpha=0.3)

box_file = os.path.join(
    desktop,
    "Dependency_Boxplot.png"
)

plt.tight_layout()
plt.savefig(box_file, dpi=300)
plt.close()

# ------------------------------------------------------------
# Final Summary
# ------------------------------------------------------------

male_mean = male.mean()
female_mean = female.mean()

print("\n")
print("="*60)
print("ANALYSIS COMPLETED SUCCESSFULLY")
print("="*60)

print(f"Number of households : {len(household):,}")
print(f"Number of individuals: {len(roster_long):,}")

print()

print(f"Male-headed dependency ratio   : {male_mean:.2f}")
print(f"Female-headed dependency ratio : {female_mean:.2f}")

print()

print(f"T-statistic : {t_stat:.4f}")
print(f"P-value     : {p_value:.6f}")

if p_value < 0.05:
    print("Interpretation: There is a statistically significant difference between male- and female-headed households (p < 0.05).")
else:
    print("Interpretation: There is no statistically significant difference between male- and female-headed households (p ≥ 0.05).")

print()

print("OUTPUT FILES CREATED")
print("-"*60)
print("✓ Household_Roster_Long.xlsx")
print("✓ Cleaned_Household_Data.xlsx")
print("✓ Overall_Statistics.xlsx")
print("✓ Grouped_Statistics.xlsx")
print("✓ Report_Table.xlsx")
print("✓ T_Test_Results.txt")
print("✓ Average_Dependency_Ratio.png")
print("✓ Dependency_Histogram.png")
print("✓ Dependency_Boxplot.png")

print("\nAll files have been saved to:")
print(desktop)

print("="*60)


Generating visualizations...


ANALYSIS COMPLETED SUCCESSFULLY
Number of households : 15,681
Number of individuals: 72,133

Male-headed dependency ratio   : 99.44
Female-headed dependency ratio : 124.07

T-statistic : -11.8472
P-value     : 0.000000
Interpretation: There is a statistically significant difference between male- and female-headed households (p < 0.05).

OUTPUT FILES CREATED
------------------------------------------------------------
✓ Household_Roster_Long.xlsx
✓ Cleaned_Household_Data.xlsx
✓ Overall_Statistics.xlsx
✓ Grouped_Statistics.xlsx
✓ Report_Table.xlsx
✓ T_Test_Results.txt
✓ Average_Dependency_Ratio.png
✓ Dependency_Histogram.png
✓ Dependency_Boxplot.png

All files have been saved to:
C:\Users\DELL\Desktop
